# Creating a TTD sample

In [8]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import brier_score_loss, roc_auc_score

project_root = next(
    (
        folder for folder in [Path.cwd(), *Path.cwd().parents]
        if (
            folder / "src" / "origination_scorecard" / "generator.py"
        ).is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project generator.")

sys.path.insert(0, str(project_root / "src"))

from origination_scorecard.generator import (
    GeneratorConfig,
    _make_applications,
    _make_performance,
)

training_path = (
    project_root / "data" / "KGB Modelling" / "KIGB in sample.csv"
)
if not training_path.is_file():
    raise FileNotFoundError(training_path)

---

## generating the sample

In [9]:
sample_size = 2_000
random_seed = 20_261_002

config = GeneratorConfig(
    n_accounts=sample_size,
    random_seed=random_seed,
)
rng = np.random.default_rng(random_seed)

# Same application and performance-generation functions as Notebook 1.
applications = _make_applications(config, rng)
_, generated_truth = _make_performance(applications, config, rng)

# Give the independent cohort a retrospective application period.
date_rng = np.random.default_rng(random_seed + 1)
applications["application_date"] = (
    pd.Timestamp("2025-01-01")
    + pd.to_timedelta(
        date_rng.integers(0, 181, sample_size),
        unit="D",
    )
).strftime("%Y-%m-%d")

applications["account_id"] = [
    f"RETRO-2025-{i:06d}"
    for i in range(1, sample_size + 1)
]

# Keep application inputs separate from outcomes.
application_columns = [
    "account_id",
    "application_date",
    "requested_loan_amount",
    "disposable_income",
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "time_at_address_months",
    "loan_purpose",
]

retrospective_applicants = applications[
    application_columns
].copy()

# Retain the simulated outcome separately until after scoring.
retrospective_outcomes = applications[
    ["master_account_key", "account_id"]
].merge(
    generated_truth[
        ["master_account_key", "total_missed_payments"]
    ],
    on="master_account_key",
    validate="one_to_one",
)

missed = retrospective_outcomes["total_missed_payments"]
retrospective_outcomes["observed_bad"] = missed.ge(3).astype(int)
retrospective_outcomes["observed_group"] = np.select(
    [missed.eq(0), missed.between(1, 2), missed.ge(3)],
    ["Known good", "Indeterminate", "Known bad"],
    default="Invalid",
)

if retrospective_applicants["account_id"].duplicated().any():
    raise ValueError("Duplicate retrospective applicant IDs.")
if retrospective_outcomes["observed_group"].eq("Invalid").any():
    raise ValueError("Invalid generated payment outcome.")

print(f"Retrospective applicants: {len(retrospective_applicants):,}")
display(
    retrospective_outcomes["observed_group"]
    .value_counts()
    .rename_axis("Observed group")
    .reset_index(name="Accounts")
)

Retrospective applicants: 2,000


,Observed group,Accounts
0,Known good,1880
1,Known bad,80
2,Indeterminate,40


In [10]:
model_inputs = [
    "disposable_income",
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "time_at_address_months",
    "loan_purpose",
]

if through_the_door["account_id"].duplicated().any():
    raise ValueError("Duplicate applicant IDs.")
if through_the_door[model_inputs].isna().any().any():
    raise ValueError("Missing KIGB model input.")
if through_the_door["time_at_address_months"].lt(0).any():
    raise ValueError("Negative address tenure.")

outcome_columns = {
    "target_bad", "kigb_bad", "outcome",
    "missed_payment_months", "predicted_pd", "score",
}
if outcome_columns & set(through_the_door.columns):
    raise ValueError("Outcome or existing model result found in sample.")

print("The through-the-door sample is ready for KIGB scoring.")
print("Application dates:",
      through_the_door["application_date"].min(),
      "to",
      through_the_door["application_date"].max())

The through-the-door sample is ready for KIGB scoring.
Application dates: 2026-09-01 to 2026-09-30


---

## Refitting the regression on the TTD sample

In [11]:
kigb_train = pd.read_csv(training_path)

scaled_predictors = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "log_address_tenure",
]

numeric_inputs = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "disposable_income",
    "time_at_address_months",
]

def prepare_inputs(data):
    rows = data.copy()

    for column in numeric_inputs:
        rows[column] = pd.to_numeric(
            rows[column], errors="raise"
        )

    if rows[numeric_inputs + ["loan_purpose"]].isna().any().any():
        raise ValueError("Missing KIGB model input.")
    if not np.isfinite(rows[numeric_inputs].to_numpy()).all():
        raise ValueError("Infinite KIGB model input.")
    if rows["time_at_address_months"].lt(0).any():
        raise ValueError("Negative address tenure.")

    rows["low_disposable_income_flag"] = (
        rows["disposable_income"] < 150
    ).astype(float)
    rows["log_address_tenure"] = np.log1p(
        rows["time_at_address_months"]
    )
    return rows

train_inputs = prepare_inputs(kigb_train)
retrospective_inputs = prepare_inputs(
    retrospective_applicants
)

# Scaling parameters come from the development sample only.
means = train_inputs[scaled_predictors].mean()
stds = train_inputs[scaled_predictors].std(ddof=0)

if not np.isfinite(stds).all() or (stds == 0).any():
    raise ValueError("Invalid predictor variation.")

def design_matrix(rows):
    x = (rows[scaled_predictors] - means) / stds
    x.insert(0, "const", 1.0)
    x["low_disposable_income_flag"] = rows[
        "low_disposable_income_flag"
    ]
    x["loan_purpose=education"] = (
        rows["loan_purpose"] == "education"
    ).astype(float)
    return x.astype(float)

x_train = design_matrix(train_inputs)
x_retrospective = design_matrix(
    retrospective_inputs
)[x_train.columns]

y_train = train_inputs["kigb_bad"].astype(int)

kigb_model = sm.Logit(y_train, x_train).fit(
    maxiter=200, disp=False
)
if not kigb_model.mle_retvals.get("converged", False):
    raise RuntimeError("KIGB model did not converge.")

print("Model fitted on KIGB development data only.")

Model fitted on KIGB development data only.


---

## Predict the PD and log scores (TTD) & create the benchmark score

In [12]:
evaluation = retrospective_applicants.copy()
evaluation["predicted_pd"] = np.asarray(
    kigb_model.predict(x_retrospective)
)

# Your existing score scale.
factor = 20 / np.log(2)
offset = 600 - factor * np.log(50)

def pd_to_score(probability):
    p = np.clip(
        np.asarray(probability, dtype=float),
        1e-12,
        1 - 1e-12,
    )
    return offset + factor * np.log((1 - p) / p)

evaluation["score"] = pd_to_score(
    evaluation["predicted_pd"]
)

benchmark_pd = 0.025
cutoff_score = float(pd_to_score([benchmark_pd])[0])

evaluation["decision"] = np.where(
    evaluation["score"] >= cutoff_score,
    "Accept",
    "Flag / decline",
)

# Only now attach the independently generated outcomes.
evaluation = evaluation.merge(
    retrospective_outcomes[
        [
            "account_id",
            "total_missed_payments",
            "observed_group",
            "observed_bad",
        ]
    ],
    on="account_id",
    how="left",
    validate="one_to_one",
)

if evaluation["observed_bad"].isna().any():
    raise ValueError("A retrospective outcome did not link.")

print(f"2.5% PD benchmark → minimum score {cutoff_score:.1f}")
display(
    evaluation[
        [
            "account_id", "predicted_pd", "score",
            "decision", "observed_group",
        ]
    ].head(10).style.format({
        "predicted_pd": "{:.2%}",
        "score": "{:.1f}",
    })
)

2.5% PD benchmark → minimum score 592.8


,account_id,predicted_pd,score,decision,observed_group
0,RETRO-2025-000001,3.52%,582.6,Flag / decline,Known good
1,RETRO-2025-000002,2.00%,599.5,Accept,Known good
2,RETRO-2025-000003,2.83%,589.1,Flag / decline,Known bad
3,RETRO-2025-000004,6.11%,566.0,Flag / decline,Known good
4,RETRO-2025-000005,2.34%,594.8,Accept,Known good
5,RETRO-2025-000006,2.51%,592.7,Flag / decline,Known good
6,RETRO-2025-000007,3.00%,587.4,Flag / decline,Known good
7,RETRO-2025-000008,9.30%,552.8,Flag / decline,Known good
8,RETRO-2025-000009,7.88%,558.1,Flag / decline,Indeterminate
9,RETRO-2025-000010,12.18%,544.1,Flag / decline,Known good


---

## Evaluate the actual generated results around the cutoff

In [13]:
def performance_summary(group):
    expected_bads = group["predicted_pd"].sum()
    observed_bads = int(group["observed_bad"].sum())

    return pd.Series({
        "Accounts": len(group),
        "Expected bads": expected_bads,
        "Observed bads": observed_bads,
        "Predicted bad rate": group["predicted_pd"].mean(),
        "Observed bad rate": group["observed_bad"].mean(),
        "Calibration gap": (
            group["observed_bad"].mean()
            - group["predicted_pd"].mean()
        ),
        "Observed / expected": (
            observed_bads / expected_bads
            if expected_bads > 0 else np.nan
        ),
        "Brier score": brier_score_loss(
            group["observed_bad"],
            group["predicted_pd"],
        ),
    })

accepted = evaluation.loc[
    evaluation["decision"].eq("Accept")
]
flagged = evaluation.loc[
    evaluation["decision"].eq("Flag / decline")
]

summary = pd.DataFrame({
    "All applicants": performance_summary(evaluation),
    "Accepted": performance_summary(accepted),
    "Flagged / declined": performance_summary(flagged),
}).T

display(
    summary.style.format({
        "Accounts": "{:,.0f}",
        "Expected bads": "{:.1f}",
        "Observed bads": "{:,.0f}",
        "Predicted bad rate": "{:.2%}",
        "Observed bad rate": "{:.2%}",
        "Calibration gap": "{:+.2%}",
        "Observed / expected": "{:.2f}",
        "Brier score": "{:.4f}",
    })
)

print(f"Approval rate: {len(accepted) / len(evaluation):.1%}")
print(
    "ROC AUC on this retrospective synthetic sample: "
    f"{roc_auc_score(evaluation['observed_bad'], evaluation['predicted_pd']):.3f}"
)

,Accounts,Expected bads,Observed bads,Predicted bad rate,Observed bad rate,Calibration gap,Observed / expected,Brier score
All applicants,"2,000",77.1,80,3.85%,4.00%,+0.15%,1.04,0.0383
Accepted,364,7.4,10,2.03%,2.75%,+0.71%,1.35,0.0268
Flagged / declined,"1,636",69.7,70,4.26%,4.28%,+0.02%,1.00,0.0408


Approval rate: 18.2%
ROC AUC on this retrospective synthetic sample: 0.589


---

# Comparing this to different cutoffs

In [ ]:
from sklearn.metrics import brier_score_loss

pd_cutoffs = {
    "2.00% PD cutoff": 0.0200,
    "2.25% PD cutoff": 0.0225,
}

def calibration_summary(data):
    """Compare predicted PDs with observed 3+ missed-payment outcomes."""
    expected_bads = data["predicted_pd"].sum()
    observed_bads = int(data["observed_bad"].sum())

    return {
        "Accounts": len(data),
        "Expected bads": expected_bads,
        "Observed bads": observed_bads,
        "Predicted bad rate": data["predicted_pd"].mean(),
        "Observed bad rate": data["observed_bad"].mean(),
        "Calibration gap": (
            data["observed_bad"].mean()
            - data["predicted_pd"].mean()
        ),
        "Observed / expected": (
            observed_bads / expected_bads
            if expected_bads > 0 else np.nan
        ),
        "Brier score": brier_score_loss(
            data["observed_bad"],
            data["predicted_pd"],
        ),
    }

calibration_reports = {}

for cutoff_name, cutoff_pd in pd_cutoffs.items():
    # Accept accounts below the individual PD cutoff.
    accepted = evaluation.loc[
        evaluation["predicted_pd"] < cutoff_pd
    ].copy()

    flagged = evaluation.loc[
        evaluation["predicted_pd"] >= cutoff_pd
    ].copy()

    report = pd.DataFrame({
        "All applicants": calibration_summary(evaluation),
        "Accepted": calibration_summary(accepted),
        "Flagged / declined": calibration_summary(flagged),
    }).T

    calibration_reports[cutoff_name] = report

    print(f"\n{cutoff_name}")
    print(
        f"Approval rate: {len(accepted) / len(evaluation):.1%} "
        f"({len(accepted):,} of {len(evaluation):,})"
    )

    display(
        report.style.format({
            "Accounts": "{:,.0f}",
            "Expected bads": "{:.1f}",
            "Observed bads": "{:,.0f}",
            "Predicted bad rate": "{:.2%}",
            "Observed bad rate": "{:.2%}",
            "Calibration gap": "{:+.2%}",
            "Observed / expected": "{:.2f}",
            "Brier score": "{:.4f}",
        })
    )

In [ ]:
comparison = []

for cutoff_name, cutoff_pd in pd_cutoffs.items():
    accepted = evaluation.loc[
        evaluation["predicted_pd"] < cutoff_pd
    ]

    comparison.append({
        "PD cutoff": cutoff_name,
        "Accepted accounts": len(accepted),
        "Approval rate": len(accepted) / len(evaluation),
        "Expected bads": accepted["predicted_pd"].sum(),
        "Observed bads": int(accepted["observed_bad"].sum()),
        "Predicted accepted bad rate": accepted["predicted_pd"].mean(),
        "Observed accepted bad rate": accepted["observed_bad"].mean(),
        "Calibration gap": (
            accepted["observed_bad"].mean()
            - accepted["predicted_pd"].mean()
        ),
    })

display(
    pd.DataFrame(comparison).style.format({
        "Approval rate": "{:.1%}",
        "Expected bads": "{:.1f}",
        "Predicted accepted bad rate": "{:.2%}",
        "Observed accepted bad rate": "{:.2%}",
        "Calibration gap": "{:+.2%}",
    })
)